# Fine-Tuning Open Source LLMs

We will fine tune an open source model(llama 3.2) that can estimate how much something costs, from its description.

### "THE PRICE IS RIGHT" Capstone Project

Now, we will build a model that predicts how much something costs from a description, based on a scrape of Amazon data.

1. QLoRA
2. Prompt Data and Base Model
3. Training
4. Eval

Note: From this Notebook we are going to use Google Colab for Resource(GPU), so if you have high performance hardware please continue in local otherwise run this nb in colab

# Introduction to LoRA and QLoRA

## What is LoRA?

**LoRA (Low-Rank Adaptation)** is a parameter-efficient fine-tuning (PEFT) technique used to adapt large language models (LLMs) without updating all model parameters.

Instead of modifying the original weight matrix \(W\), LoRA freezes the pretrained weights and learns two smaller matrices:

\[
W' = W + BA
\]

Where:

- \(W\) = original frozen weights  
- \(A\) and \(B\) = low-rank trainable matrices  
- \(W'\) = adapted weights after fine-tuning  

### Why LoRA?

Traditional fine-tuning updates billions of parameters, requiring:

- High GPU memory
- Large storage for checkpoints
- Long training times

LoRA reduces this cost by training only a small number of additional parameters.

### Advantages

- Memory efficient
- Faster training
- Smaller checkpoints
- Easy task switching using adapters

### Typical Workflow

1. Load pretrained model
2. Freeze original weights
3. Insert LoRA adapters into attention layers
4. Train only adapter weights
5. Merge adapters if needed for inference

## What is QLoRA?

**QLoRA (Quantized LoRA)** extends LoRA by combining:

- **4-bit quantization**
- **Low-rank adapters (LoRA)**

This allows fine-tuning very large models on consumer GPUs.

### Core Idea

QLoRA:

1. Quantizes the base model to 4-bit precision
2. Keeps quantized weights frozen
3. Trains LoRA adapters in higher precision

### Benefits of QLoRA

| Feature | LoRA | QLoRA |
|---|---|---|
| Memory Usage | Low | Extremely Low |
| Quantization | No | 4-bit |
| GPU Requirement | Moderate | Consumer GPUs |
| Training Speed | Fast | Fast |
| Model Quality | High | Near full fine-tuning |

### Key Technologies in QLoRA

- **NF4 Quantization**: Optimized 4-bit representation
- **Double Quantization**: Compresses quantization constants
- **Paged Optimizers**: Prevents GPU memory spikes

### Why QLoRA Matters

QLoRA made it possible to fine-tune models like:

- LLaMA
- Mistral
- Falcon

on GPUs with limited VRAM (e.g., 16GB–24GB).

## Summary

- **LoRA** reduces trainable parameters using low-rank matrices.
- **QLoRA** further reduces memory usage using 4-bit quantization.
- Both are widely used for efficient LLM fine-tuning.

In [ ]:
# imports Libraries

import os
import re
import math
import gc
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel
from datetime import datetime

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"

LITE_MODE = False

DATA_USER = "Arivukkarasu"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"


FINETUNED_MODEL = f"ed-donner/price-2025-11-30_15.10.55-lite"

### Log in to HuggingFace

If you don't already have a HuggingFace account, visit https://huggingface.co to sign up and create a token.

Then select the Secrets for this Notebook by clicking on the key icon in the left, and add a new secret called `HF_TOKEN` with the value as your token.

In [ ]:
# Log in to HuggingFace

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Trying out different Quantization

In [ ]:
# Load the Base Model without quantization

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")

In [ ]:
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

In [ ]:
base_model

## Deleting the model to clear GPU Space

In order to load the next model and clear out the cache of the last model

In [ ]:
# Delete model and related objects
del base_model

# Run Python garbage collection
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()

# Optional: release inter-process cached memory
torch.cuda.ipc_collect()

In [ ]:
# # Else please use this function
# def free_gpu_memory(model):
#     del model
#     gc.collect()

#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()
#         torch.cuda.ipc_collect()

#     print("GPU memory cleaned")

# free_gpu_memory(base_model)

In [ ]:
print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

In [ ]:
# Load the Base Model using 8 bit

quant_config = BitsAndBytesConfig(load_in_8bit=True)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

In [ ]:
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

In [ ]:
base_model

In [ ]:
# Delete model and related objects
del base_model

# Run Python garbage collection
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()

# Optional: release inter-process cached memory
torch.cuda.ipc_collect()

print(torch.cuda.memory_allocated() / 1024**3, "GB allocated")
print(torch.cuda.memory_reserved() / 1024**3, "GB reserved")

In [ ]:
# Load the Tokenizer and the Base Model using 4 bit

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

In [ ]:
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.2f} GB")

In [ ]:
base_model

In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, FINETUNED_MODEL)

In [ ]:
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e9:,.2f} GB")

In [ ]:
fine_tuned_model

### Understanding LoRA weights and dimensions

#### More common QLoRA setup that we'll use for the Lite Training `LITE_MODE=True`

r = 32, which is a common value for r

Target modules are the Attention layers only

In [ ]:
# Each of the Target Modules has 2 LoRA Adaptor matrices, called lora_A and lora_B
# These are designed so that weights can be adapted by adding alpha * lora_A * lora_B
# Let's count the number of weights using their dimensions:

r = 32

# See the matrix dimensions above

# Attention layers
lora_q_proj = 3072 * r + 3072 * r
lora_k_proj = 3072 * r + 1024 * r
lora_v_proj = 3072 * r + 1024 * r
lora_o_proj = 3072 * r + 3072 * r

# Each layer comes to
lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj

# There are 28 layers
params = lora_layer * 28

# So the total size in MB is
size = (params * 4) / 1_000_000

print(f"Total number of params: {params:,} and size {size:,.1f}MB")

### And now, for training with the Full Dataset `LITE_MODE=False`

This is a pretty extreme LoRA setup. We have so much training data that I'm trying to train a lot!

We use 256 dimensions in the LoRA matrices and we have LoRA matrices for the attention layers and the MLP layers:

In [ ]:
# Each of the Target Modules has 2 LoRA Adaptor matrices, called lora_A and lora_B
# These are designed so that weights can be adapted by adding alpha * lora_A * lora_B
# Let's count the number of weights using their dimensions:

r = 256

# See the matrix dimensions above

# Attention layers
lora_q_proj = 3072 * r + 3072 * r
lora_k_proj = 3072 * r + 1024 * r
lora_v_proj = 3072 * r + 1024 * r
lora_o_proj = 3072 * r + 3072 * r

# MLP layers
lora_gate_proj = 3072 * r + 8192 * r
lora_up_proj = 3072 * r + 8192 * r
lora_down_proj = 3072 * r + 8192 * r

# Each layer comes to
lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj + lora_gate_proj + lora_up_proj + lora_down_proj

# There are 28 layers
params = lora_layer * 28

# So the total size in MB is
size = (params * 4) / 1_000_000

print(f"Total number of params: {params:,} and size {size:,.1f}MB")